In [1]:
import customtkinter as ctk
import threading
import time
import numpy as np
import sounddevice as sd
import soundfile as sf
import serial
import serial.tools.list_ports
import os
import librosa
import torch
import torch.nn as nn
from torchaudio import transforms
from google import genai

# --- MATPLOTLIB ---
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import matplotlib
matplotlib.use("TkAgg")

# ==========================================
# 🎨 AYARLAR
# ==========================================
ctk.set_appearance_mode("Dark")
ctk.set_default_color_theme("green")

class Config:
    API_KEY = ""
    ARDUINO_PORT = "COM10"
    BAUD_RATE = 9600
    SAMPLE_RATE = 16000
    
    FILE_NORMAL = "normal.wav"
    FILE_ANOMALY = "anormal.wav"
    MODEL_PATH = "fan_model_minus6db.pth"

    # Renkler
    COLOR_BG = "#121212"
    COLOR_CARD = "#1E1E1E"
    COLOR_GREEN = "#00E676"
    COLOR_RED = "#D50000"
    COLOR_CYAN = "#00B0FF"
    COLOR_YELLOW = "#FFD600"
    COLOR_CURSOR = "#FFFFFF"
    FONT_MAIN = "Roboto Medium"
    FONT_MONO = "Consolas"

# ==========================================
# 🧠 1. PYTORCH MODELİ
# ==========================================
class FanModel(nn.Module):
    def __init__(self):
        super(FanModel, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) 
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

# ==========================================
# 🤖 2. GEMINI CLIENT
# ==========================================
class GeminiAgent:
    def __init__(self):
        self.client = None
        try:
            self.client = genai.Client(api_key=Config.API_KEY)
            print("✅ Gemini AI Hazır.")
        except: pass

    def ask(self, prompt):
        if not self.client: return "API Bağlantısı Yok."
        try:
            response = self.client.models.generate_content(
                model="gemini-1.5-flash", contents=prompt
            )
            return response.text
        except Exception as e: return f"Hata: {str(e)}"

# ==========================================
# 🔌 3. DONANIM
# ==========================================
class HardwareManager:
    def __init__(self):
        self.ser = None
        self.connect()

    def connect(self):
        try:
            self.ser = serial.Serial(Config.ARDUINO_PORT, Config.BAUD_RATE, timeout=1)
            time.sleep(2)
            self.set_motor(False)
        except: print("⚠️ Arduino Yok (Simülasyon)")

    def set_motor(self, state):
        if self.ser and self.ser.is_open:
            cmd = "M1\n" if state else "M0\n"
            try: self.ser.write(cmd.encode())
            except: pass

# ==========================================
# 🎧 4. ANALİZ MOTORU
# ==========================================
class AnalysisEngine:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self._load_model()
        
        self.audio_cache = {}
        self._load_file_to_cache(Config.FILE_NORMAL)
        self._load_file_to_cache(Config.FILE_ANOMALY)

        self.mel_transform = transforms.MelSpectrogram(
            sample_rate=Config.SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512
        ).to(self.device)
        self.db_transform = transforms.AmplitudeToDB().to(self.device)

    def _load_model(self):
        try:
            model = FanModel().to(self.device)
            if os.path.exists(Config.MODEL_PATH):
                model.load_state_dict(torch.load(Config.MODEL_PATH, map_location=self.device))
                model.eval()
                return model
        except: pass
        return None

    def _load_file_to_cache(self, path):
        if os.path.exists(path):
            y, sr = librosa.load(path, sr=Config.SAMPLE_RATE, mono=False)
            if y.ndim > 1: y = y[0, :]
            self.audio_cache[path] = y

    def get_audio_data(self, path):
        return self.audio_cache.get(path, np.zeros(100))

    def predict(self, file_path):
        if not self.model: return True, 0.99
        try:
            y = self.audio_cache.get(file_path)
            if y is None: return False, 0.0

            waveform = torch.tensor(y).float().unsqueeze(0)
            target = Config.SAMPLE_RATE * 3
            if waveform.shape[1] > target: waveform = waveform[:, :target]
            else: waveform = torch.nn.functional.pad(waveform, (0, target - waveform.shape[1]))
            
            spec = self.mel_transform(waveform.to(self.device))
            spec = self.db_transform(spec).unsqueeze(0)
            spec = torch.nn.functional.interpolate(spec, size=(128, 128), mode='bilinear')
            spec = (spec - spec.min()) / (spec.max() - spec.min() + 1e-6)
            
            with torch.no_grad():
                prob = self.model(spec).item()
            return prob > 0.5, prob
        except: return False, 0.0

# ==========================================
# 📊 5. DETAYLI GRAFİK (ZOOM & HAREKET)
# ==========================================
class AdvancedWaveformDisplay(ctk.CTkFrame):
    def __init__(self, parent):
        super().__init__(parent, fg_color=Config.COLOR_CARD, corner_radius=15)
        self.pack(fill="x", padx=20, pady=10)
        
        # Grafik boyutu
        self.fig, self.ax = plt.subplots(figsize=(8, 2.5), dpi=100)
        self.fig.patch.set_facecolor(Config.COLOR_CARD) 
        self.ax.set_facecolor(Config.COLOR_BG)          
        
        # Kenarlıkları kaldır
        for spine in self.ax.spines.values():
            spine.set_visible(False)
        self.ax.get_xaxis().set_ticks([])
        self.ax.get_yaxis().set_ticks([])

        self.canvas = FigureCanvasTkAgg(self.fig, master=self)
        self.canvas.get_tk_widget().pack(fill="both", expand=True, padx=10, pady=10)
        
        # Çizgi ve Cursor nesneleri
        self.line, = self.ax.plot([], [], color=Config.COLOR_CYAN, linewidth=1, alpha=0.9)
        self.cursor = self.ax.axvline(x=0, color=Config.COLOR_CURSOR, linewidth=2, alpha=1)

    def set_waveform(self, audio_data, color):
        self.ax.clear() 
        self.ax.set_facecolor(Config.COLOR_BG)
        
        # Veri Hazırlama (Daha az seyreltme = Daha yüksek çözünürlük)
        downsampled = audio_data[::5] # Sadece her 5. veriyi al (Performans/Kalite dengesi)
        
        x = np.arange(len(downsampled))
        
        # Alanı Doldur (Fill)
        self.ax.fill_between(x, downsampled, color=color, alpha=0.4)
        self.ax.plot(x, downsampled, color=color, linewidth=0.8)
        
        # Cursor'ı başa al
        self.cursor = self.ax.axvline(x=0, color="white", linewidth=2)
        
        self.ax.set_xlim(0, len(downsampled))
        
        # 🔥 ÖNEMLİ: ZOOM AYARI 🔥
        # Normalde -1 ile 1 arasıdır. -0.15 ile 0.15 yaparak dalgaları devasa gösteriyoruz.
        self.ax.set_ylim(-0.15, 0.15) 
        
        self.canvas.draw()
        return len(downsampled)

    def move_cursor(self, current_pos, total_len):
        self.cursor.set_xdata([current_pos, current_pos])
        self.canvas.draw()

# ==========================================
# 🖥️ 6. UI
# ==========================================
class ResultCard(ctk.CTkFrame):
    def __init__(self, parent, title):
        super().__init__(parent, fg_color=Config.COLOR_CARD, corner_radius=15)
        self.pack(side="left", fill="both", expand=True, padx=10, pady=10)
        self.lbl_title = ctk.CTkLabel(self, text=title, font=(Config.FONT_MAIN, 14, "bold"), text_color="gray")
        self.lbl_title.pack(pady=(15, 5))
        self.lbl_res = ctk.CTkLabel(self, text="---", font=(Config.FONT_MAIN, 20, "bold"), text_color="white")
        self.lbl_res.pack(pady=5)
        self.bar = ctk.CTkProgressBar(self, height=8)
        self.bar.set(0)
        self.bar.pack(pady=(10, 15), padx=30, fill="x")

class SonusApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.hw = HardwareManager()
        self.ai = GeminiAgent()
        self.engine = AnalysisEngine()
        self.title("SONUS| Otonom Arıza Tanı")
        self.geometry("1100x700") # Boyutu optimize ettik
        self.create_ui()

    def create_ui(self):
        # SCROLLABLE FRAME
        self.scroll_frame = ctk.CTkScrollableFrame(self, fg_color="transparent")
        self.scroll_frame.pack(fill="both", expand=True)

        # HEADER
        header = ctk.CTkFrame(self.scroll_frame, height=80, fg_color="transparent")
        header.pack(fill="x", padx=30, pady=20)
        ctk.CTkLabel(header, text="SONUS AI MONITOR", font=(Config.FONT_MAIN, 28, "bold")).pack(side="left")
        self.status_badge = ctk.CTkButton(header, text="● SİSTEM HAZIR", fg_color="#333", hover=False, 
                                          text_color="white", corner_radius=20, width=120)
        self.status_badge.pack(side="right")

        # CARDS
        cards_frame = ctk.CTkFrame(self.scroll_frame, fg_color="transparent")
        cards_frame.pack(fill="x", padx=20, pady=5)
        self.card_1 = ResultCard(cards_frame, "TEST 1: NORMAL SES")
        self.card_2 = ResultCard(cards_frame, "TEST 2: ANORMAL SES")

        # GRAPHIC
        ctk.CTkLabel(self.scroll_frame, text="YÜKSEK HASSASİYETLİ SPEKTRUM", font=(Config.FONT_MAIN, 12, "bold"), text_color="gray").pack(anchor="w", padx=30, pady=(10,0))
        self.waveform = AdvancedWaveformDisplay(self.scroll_frame)

        # TERMINAL
        term_frame = ctk.CTkFrame(self.scroll_frame, fg_color="#000", corner_radius=15, height=200)
        term_frame.pack(fill="x", padx=30, pady=10)
        self.log_box = ctk.CTkTextbox(term_frame, font=(Config.FONT_MONO, 12), fg_color="transparent", text_color="#00E676", height=150)
        self.log_box.pack(fill="both", expand=True, padx=10, pady=10)

        # START BUTTON
        self.btn_start = ctk.CTkButton(self.scroll_frame, text="▶ SENARYOYU BAŞLAT", command=self.start_thread,
                                       height=60, font=(Config.FONT_MAIN, 16, "bold"), 
                                       fg_color=Config.COLOR_GREEN, hover_color="#00A040")
        self.btn_start.pack(fill="x", padx=30, pady=20)

    def log(self, msg):
        self.log_box.insert("end", f"> {msg}\n")
        self.log_box.see("end")

    def start_thread(self):
        threading.Thread(target=self.run_scenario, daemon=True).start()

    def play_and_animate(self, audio_data, graph_len):
        duration = len(audio_data) / Config.SAMPLE_RATE
        play_data = np.column_stack((audio_data, audio_data))
        sd.play(play_data, Config.SAMPLE_RATE)
        start_time = time.time()
        
        while True:
            elapsed = time.time() - start_time
            if elapsed >= duration:
                break
            progress = elapsed / duration
            cursor_pos = int(progress * graph_len)
            self.waveform.move_cursor(cursor_pos, graph_len)
            time.sleep(0.04) # 25 FPS akıcılık
        sd.stop()

    def run_scenario(self):
        self.btn_start.configure(state="disabled", text="ANALİZ SÜRÜYOR...")
        self.log_box.delete("0.0", "end")
        self.status_badge.configure(text="● RUNNING", fg_color=Config.COLOR_GREEN)

        if not os.path.exists(Config.FILE_NORMAL):
            self.log("HATA: Dosyalar eksik!")
            return

        self.log("Sistem Başlatıldı. Motor: ON")
        self.hw.set_motor(True)
        time.sleep(1)

        # --- TEST 1: NORMAL ---
        self.log("ADIM 1: Normal ses analiz ediliyor...")
        audio_data_normal = self.engine.get_audio_data(Config.FILE_NORMAL)
        
        # Grafiği Hazırla (Mavi - Analiz Modu)
        graph_len = self.waveform.set_waveform(audio_data_normal, color=Config.COLOR_CYAN)
        
        # Çal ve Oynat
        self.play_and_animate(audio_data_normal, graph_len)
        
        # Sonuç
        prob1 = self.engine.predict(Config.FILE_NORMAL)[1]
        
        # BİTTİKTEN SONRA YEŞİL YAP
        self.waveform.set_waveform(audio_data_normal, color=Config.COLOR_GREEN)
        
        self.card_1.lbl_res.configure(text=f"%{prob1*100:.1f} NORMAL", text_color=Config.COLOR_GREEN)
        self.card_1.bar.set(prob1)
        self.card_1.bar.configure(progress_color=Config.COLOR_GREEN)
        self.log(f"Sonuç: Normal (%{prob1*100:.1f})")
        
        time.sleep(1)

        # --- TEST 2: ANORMAL ---
        self.log("ADIM 2: Arıza simülasyonu başlatılıyor...")
        audio_data_anomaly = self.engine.get_audio_data(Config.FILE_ANOMALY)
        
        # Grafiği Hazırla (Sarı - Analiz Modu)
        graph_len = self.waveform.set_waveform(audio_data_anomaly, color=Config.COLOR_YELLOW)
        
        # Çal ve Oynat
        self.play_and_animate(audio_data_anomaly, graph_len)
        
        # Sonuç
        is_anom, prob2 = self.engine.predict(Config.FILE_ANOMALY)
        
        # BİTTİKTEN SONRA KIRMIZI YAP (Eğer Hata Varsa)
        if is_anom:
            self.waveform.set_waveform(audio_data_anomaly, color=Config.COLOR_RED)
        
        self.card_2.lbl_res.configure(text=f"%{prob2*100:.1f} ARIZA", text_color=Config.COLOR_RED)
        self.card_2.bar.set(prob2)
        self.card_2.bar.configure(progress_color=Config.COLOR_RED)
        self.log(f"Sonuç: ARIZA TESPİT EDİLDİ (%{prob2*100:.1f})")

        if is_anom:
            self.hw.set_motor(False)
            self.status_badge.configure(text="● CRITICAL", fg_color=Config.COLOR_RED)
            self.log("!!! MOTOR ACİL DURDURULDU !!!")
            
            self.log("Gemini AI Rapor Hazırlıyor...")
            prompt = f"Fan motorunda %{prob2*100:.1f} oranında mekanik anomali var. Teknik analiz yap."
            report = self.ai.ask(prompt)
            self.log("-" * 40)
            self.log(report)
            self.log("-" * 40)
        
        self.btn_start.configure(state="normal", text="▶ SENARYOYU TEKRARLA")

if __name__ == "__main__":
    app = SonusApp()
    app.mainloop()

⚠️ Arduino Yok (Simülasyon)
✅ Gemini AI Hazır.


Exception in thread Thread-3 (run_scenario):
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\hanal\AppData\Local\Temp\ipykernel_39684\3516651073.py", line 328, in run_scenario
  File "C:\Users\hanal\AppData\Local\Temp\ipykernel_39684\3516651073.py", line 303, in play_and_animate
  File "C:\Users\hanal\AppData\Local\Temp\ipykernel_39684\3516651073.py", line 222, in move_cursor
  File "c:\Users\hanal\OneDrive\Belgeler\DeepLearningWithTorch\.venv\Lib\site-packages\matplotlib\backends\backend_tkagg.py", line 11, in draw
    self.blit()
  File "c:\Users\hanal\OneDrive\Belgeler\DeepLearningWithTorch\.venv\Lib\site-packages\matplotlib\ba

KeyboardInterrupt: 